# The discrete Hankel transform

The discrete Hankel transform (DHT) is the radial step of the polar Fourier transform: after the angular DFT has split a function into harmonics, each harmonic goes through its own DHT. This notebook introduces the continuous Hankel transform, the discrete transform PyPFT implements, the role of the space limit $R$, and the transform rules the discrete transform obeys.

`pypft.hankel_transform` and `pypft.inverse_hankel_transform` compute the DHT of order $n$ for a space limit $R$, and `pypft.sample_points` returns the radial and frequency points the transform expects its samples at.

In [ ]:
# NumPy for arrays, matplotlib for figures, SciPy for Bessel functions, and PyPFT itself
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import jn_zeros, jv

import pypft

## The continuous Hankel transform

The Hankel transform of order $n$ maps a function $f(r)$ of a radius $r \ge 0$ to a function $F_n(\rho)$ of a radial frequency $\rho \ge 0$ [[1]](#ref-1) (Eqs. 1-2):

$$F_n(\rho) = \int_0^\infty f(r)\, J_n(\rho r)\, r \, dr, \qquad f(r) = \int_0^\infty F_n(\rho)\, J_n(\rho r)\, \rho \, d\rho,$$

where $J_n$ is the Bessel function of the first kind of order $n$. The same formula transforms in both directions: the Hankel transform is its own inverse.

It is the radial part of a 2-D Fourier transform. Take a function with a single angular harmonic, $g(r, \theta) = f(r)\, e^{in\theta}$, and its 2-D Fourier transform $G(\boldsymbol{\rho}) = \iint g(\mathbf{x})\, e^{-i \boldsymbol{\rho} \cdot \mathbf{x}} \, d\mathbf{x}$, with the frequency $\boldsymbol{\rho}$ written in polar coordinates $(\rho, \psi)$. Expanding $e^{-i\rho r \cos(\theta - \psi)}$ in Bessel functions (the Jacobi-Anger expansion) and integrating over $\theta$ leaves a single term:

$$G(\rho, \psi) = 2\pi\, i^{-n}\, e^{in\psi}\, F_n(\rho).$$

The harmonic keeps its angular dependence, and its radial profile goes through a Hankel transform of the same order. The factor $2\pi\, i^{-n}$ is exactly what `forward_pft` multiplies each harmonic's Hankel transform by.

The discrete Hankel transform is not just a quadrature rule for this integral. Baddour defines it as a transform in its own right, a matrix acting on $N - 1$ samples, with its own inverse and its own transform rules [[1]](#ref-1) (Eqs. 39, 50-51), which also approximates the continuous transform at particular sample points [[1]](#ref-1) (Eqs. 42-43). The rest of this notebook introduces those sample points, the matrix, and its rules.

## The space limit R

The DHT assumes the function is **space-limited**: $f(r) = 0$ for $r > R$. The space limit $R$ has the same length unit as `r`, whatever unit the data uses (for example millimeters). The kernel $J_n(\rho r)$ needs a dimensionless argument, so $\rho$ is in radians per that unit: an angular frequency, with $\rho / 2\pi$ in cycles per unit length.

$R$ fixes where the $N - 1$ samples of an order-$n$ transform sit, in both domains [[1]](#ref-1) (Eq. 31):

$$r_{nk} = \frac{j_{nk}\, R}{j_{nN}}, \qquad \rho_{nk} = \frac{j_{nk}}{R}, \qquad k = 1, \dots, N - 1,$$

where $j_{nk}$ is the $k$-th positive zero of $J_n$. The largest frequency sample approaches the band limit $W_\rho = j_{nN}/R$. Consecutive Bessel zeros are roughly $\pi$ apart, so the frequency samples are roughly $\pi / R$ apart and the radial samples roughly $\pi R / j_{nN}$ apart.

This makes $R$ a real choice, not just a unit:

- **Too small**, and the part of $f$ beyond $R$ is cut off: the transform is computed for a truncated function.
- **Too large**, and a fixed number of samples is spread over mostly empty space: the radial samples get coarser and the band limit $W_\rho$ drops, so fine detail is lost.

The figure shows the sample points of an order-0 transform with 32 samples, for two space limits. Doubling $R$ stretches the radial samples to twice the range and squeezes the frequency samples into half of it.

In [ ]:
# An order-0 transform with 32 samples
n, size = 0, 32

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))
for R in (4.0, 8.0):
    # Radial and frequency sample points for this space limit
    r, rho = pypft.sample_points(n=n, size=size, R=R)
    k = np.arange(start=1, stop=size + 1)

    # Sample positions against their index, in both domains
    axes[0].plot(k, r, marker="o", markersize=3, label=f"$R = {R:g}$")
    axes[1].plot(k, rho, marker="o", markersize=3, label=f"$R = {R:g}$")
axes[0].set_xlabel(xlabel="index $k$")
axes[0].set_ylabel(ylabel="$r_{0k}$ (length unit)")
axes[0].set_title(label="radial samples")
axes[1].set_xlabel(xlabel="index $k$")
axes[1].set_ylabel(ylabel=r"$\rho_{0k}$ (rad per length unit)")
axes[1].set_title(label="frequency samples")
for ax in axes:
    ax.legend()
fig.tight_layout()
plt.show()

## A Gaussian that transforms into itself

A Gaussian is the standard first check of a Hankel transform, because its transform is known exactly. For order 0,

$$\int_0^\infty e^{-r^2 / (2\sigma^2)}\, J_0(\rho r)\, r \, dr = \sigma^2 e^{-\sigma^2 \rho^2 / 2},$$

which is the 2-D Fourier transform of a circularly symmetric Gaussian divided by $2\pi$ (the $n = 0$ case of the relation above). With $\sigma = 1$ the two sides have the same shape and the same height: $e^{-r^2/2}$ transforms into $e^{-\rho^2/2}$. In this convention, with no $2\pi$ inside the Bessel function, $\sigma = 1$ is the only width for which the Gaussian is its own transform.

The discrete transform reproduces the continuous one when both $f$ and $F$ are negligible outside the sampled ranges, $r \le R$ and $\rho \le W_\rho$. With $R = 8$ the Gaussian has dropped to $e^{-32} \approx 10^{-14}$ at the space limit, and 64 samples put the band limit near $\rho = 25$, far beyond where $e^{-\rho^2/2}$ matters.

In [ ]:
# Order 0, space limit 8, and 64 samples
n, R, size = 0, 8.0, 64
r, rho = pypft.sample_points(n=n, size=size, R=R)

# Sample the Gaussian at the radial points and transform it
f = np.exp(-(r**2) / 2)
F = pypft.hankel_transform(f=f, n=n, R=R)

# The exact continuous transform, evaluated at the frequency points
F_expected = np.exp(-(rho**2) / 2)
error = np.abs(F - F_expected)

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(13, 3.8))

# Left: the samples of f
axes[0].plot(r, f, marker="o", markersize=3)
axes[0].set_xlabel(xlabel="$r$")
axes[0].set_title(label="$f(r) = e^{-r^2/2}$")

# Middle: the computed and the exact transform
axes[1].plot(rho, F, marker="o", markersize=3, label="DHT")
axes[1].plot(rho, F_expected, color="black", linestyle="--", label="exact")
axes[1].set_xlim(left=0.0, right=8.0)
axes[1].set_xlabel(xlabel=r"$\rho$")
axes[1].set_title(label=r"$F(\rho)$")
axes[1].legend()

# Right: the absolute error at every frequency sample, on a log scale
axes[2].semilogy(rho, error, marker="o", markersize=3)
axes[2].set_xlabel(xlabel=r"$\rho$")
axes[2].set_ylim(bottom=1e-18, top=1e-12)
axes[2].set_title(label="absolute error")
fig.tight_layout()
plt.show()

print(f"maximum absolute error: {error.max():.2e}")

## The DHT as a matrix

For $N - 1$ samples of order $n$, the DHT is the $(N - 1) \times (N - 1)$ matrix [[1]](#ref-1) (Eq. 39)

$$Y^{nN}_{mk} = \frac{2}{j_{nN}\, J_{n+1}^2(j_{nk})}\, J_n\!\left(\frac{j_{nm}\, j_{nk}}{j_{nN}}\right), \qquad 1 \le m, k \le N - 1.$$

Applied to the samples $f(r_{nk})$, and scaled by $R$, it gives the transform at the frequency samples, and the same matrix gives the inverse [[1]](#ref-1) (Eqs. 42-43):

$$F(\rho_{nm}) \approx \frac{R^2}{j_{nN}} \sum_{k=1}^{N-1} Y^{nN}_{mk}\, f(r_{nk}), \qquad f(r_{nk}) \approx \frac{j_{nN}}{R^2} \sum_{m=1}^{N-1} Y^{nN}_{km}\, F(\rho_{nm}).$$

This is exactly what `hankel_transform` and `inverse_hankel_transform` compute. The helper below builds $Y^{nN}$ straight from its definition, so the rest of this notebook can compare the transform with its kernel. `pypft.sample_points` and `hankel_transform` take the number of samples, `size`, which is $N - 1$.

In [ ]:
def dht_kernel(order, size):
    """Build the DHT matrix of an integer order from its definition.

    Returns the matrix and the zero j_N that scales the physical transform.
    """
    # The size + 1 first positive zeros of the Bessel function of this order
    zeros = jn_zeros(n=abs(order), nt=size + 1)
    j_k, j_N = zeros[:-1], zeros[-1]

    # J_n(j_m j_k / j_N), with column k divided by j_N J_(n+1)(j_k)^2 / 2
    column_scale = 2 / (j_N * jv(order + 1, j_k) ** 2)
    return column_scale * jv(order, np.outer(a=j_k, b=j_k) / j_N), j_N


# The public transform is the kernel, scaled by R^2 / j_N
Y, j_N = dht_kernel(order=n, size=size)
F_from_kernel = (R**2 / j_N) * (Y @ f)
print(f"hankel_transform vs. scaled kernel: {np.abs(F - F_from_kernel).max():.1e}")

# The kernel itself: each column has its own scale, so it is not symmetric
fig, ax = plt.subplots(figsize=(4.5, 4))
shown = ax.imshow(X=Y, cmap="RdBu_r")
ax.set_xlabel(xlabel="column $k$")
ax.set_ylabel(ylabel="row $m$")
ax.set_title(label=f"$Y^{{nN}}$, $n = {n}$, $N - 1 = {size}$")
fig.colorbar(mappable=shown, ax=ax)
plt.show()

## The kernel is its own inverse

The matrix is not symmetric, because its column scale $1 / J_{n+1}^2(j_{nk})$ depends on the column only, yet it is its own inverse [[1]](#ref-1) (Eq. 41):

$$Y^{nN} Y^{nN} = I.$$

That is why the forward and inverse transforms above use the same matrix and differ only by the scalar in front. The identity rests on a discrete orthogonality relation of Bessel functions that is exact only in the limit of many samples [[1]](#ref-1) (Eq. 37). For a few tens of samples the residual $Y^{nN} Y^{nN} - I$ is already small, and it grows slowly with the order. A forward-then-inverse round trip therefore returns any signal, not only a smooth one, to within that residual.

In [ ]:
# A higher order and a random signal: nothing smooth about it
n, R, size = 2, 8.0, 32
Y, j_N = dht_kernel(order=n, size=size)
rng = np.random.default_rng(seed=0)
signal = rng.standard_normal(size=size)

# Forward, then inverse, through the public API
transformed = pypft.hankel_transform(f=signal, n=n, R=R)
reconstructed = pypft.inverse_hankel_transform(F=transformed, n=n, R=R)

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))

# Left: the kernel times itself is the identity
shown = axes[0].imshow(X=Y @ Y, cmap="gray_r")
axes[0].set_title(label="$Y^{nN} Y^{nN}$")
fig.colorbar(mappable=shown, ax=axes[0])

# Right: the random signal and its round trip lie on top of each other
axes[1].plot(signal, marker="o", label="signal")
axes[1].plot(reconstructed, marker="x", linestyle="none", label="round trip")
axes[1].set_xlabel(xlabel="sample index")
axes[1].legend()
fig.tight_layout()
plt.show()

print(f"max |Y Y - I|:          {np.abs(Y @ Y - np.eye(N=size)).max():.1e}")
print(f"max round-trip error:   {np.abs(reconstructed - signal).max():.1e}")

## The Kronecker-delta pair

The claim: a discrete delta at index $k_0$, the vector $\delta_{kk_0}$ that is $1$ at $k = k_0$ and $0$ elsewhere, transforms into the $k_0$-th column of the kernel [[1]](#ref-1) (Eq. 65),

$$\delta_{kk_0} \;\Longleftrightarrow\; Y^{nN}_{mk_0}.$$

This is just the matrix product picking out one column. With the physical transform's scale, `hankel_transform` of a delta is that column times $R^2 / j_{nN}$. Unlike the DFT of a delta, a single complex exponential of constant magnitude, the result is a full, oscillating vector.

Because the kernel is its own inverse, the pair also runs the other way: the $k_0$-th column transforms back into the delta [[1]](#ref-1) (Eq. 68).

In [ ]:
# A discrete delta at Python index k0 (the formulas count from 1, Python from 0)
k0 = 10
delta = np.zeros(shape=size)
delta[k0] = 1.0

# Its transform, next to the explicitly built, scaled kernel column
delta_transform = pypft.hankel_transform(f=delta, n=n, R=R)
kernel_column = (R**2 / j_N) * Y[:, k0]

# The inverse transform of the result: the delta again
delta_back = pypft.inverse_hankel_transform(F=delta_transform, n=n, R=R)

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 3.8))

# Left: the transform of the delta is the kernel's k0-th column
axes[0].plot(kernel_column, color="black", label="$(R^2 / j_{nN})\\, Y_{m k_0}$")
axes[0].plot(
    delta_transform, marker="o", linestyle="none", label="hankel_transform(delta)"
)
axes[0].set_xlabel(xlabel="frequency index $m$")
axes[0].set_title(label=f"forward, Python index k0 = {k0}")
axes[0].legend()

# Right: transforming back recovers the delta
axes[1].stem(delta_back)
axes[1].set_xlabel(xlabel="sample index $k$")
axes[1].set_title(label="inverse of the result")
fig.tight_layout()
plt.show()

column_mismatch = np.abs(delta_transform - kernel_column).max()
print(f"transform vs. kernel column: {column_mismatch:.1e}")
print(f"round trip vs. delta:        {np.abs(delta_back - delta).max():.1e}")

## Negative orders

The polar Fourier transform needs Hankel transforms of negative orders too, one for every negative harmonic. They need no kernel of their own:

$$Y^{(-n)N} = (-1)^n\, Y^{nN}.$$

This follows from the exact Bessel identity $J_{-n}(x) = (-1)^n J_n(x)$, true for every $x$. The zeros of $J_{-n}$ are those of $J_n$, so the sample points are unchanged. The numerator of the kernel picks up the factor $(-1)^n$. The denominator is unchanged: $J_{-n+1}(j_{nk}) = \pm J_{n-1}(j_{nk})$, and at a zero of $J_n$ the recurrence $J_{n-1}(x) + J_{n+1}(x) = (2n/x) J_n(x)$ gives $J_{n-1}(j_{nk}) = -J_{n+1}(j_{nk})$, so the squares agree. `hankel_transform` therefore accepts only $n \ge 0$, and the polar transform multiplies the sign in itself.

`dht_kernel` above builds the kernel for any integer order, including negative ones, straight from the definition:

In [ ]:
for order in (1, 2, 3, 4):
    # The kernels of orders -n and n, each built from its own definition
    Y_positive, _ = dht_kernel(order=order, size=size)
    Y_negative, _ = dht_kernel(order=-order, size=size)

    # They differ exactly by the sign (-1)^n
    mismatch = np.abs(Y_negative - (-1) ** order * Y_positive).max()
    print(f"n = {order}: max |Y^(-n) - (-1)^n Y^n| = {mismatch:.1e}")

## The generalized shift

The DFT has a shift theorem: shifting a sequence, $f_{k - k_0}$, multiplies its transform by a complex exponential. The DHT cannot copy it. The shifted index $k - k_0$ falls outside $1, \dots, N - 1$ for some $k$, and the DFT gets around that only because its kernel is periodic, so the index can wrap around modulo $N$. The Bessel kernel is not periodic, and $J_n(x - y)$ has no simple relation to $J_n(x)$, so $f_{k - k_0}$ has no useful meaning for the DHT.

Baddour's way out follows the continuous shift theorem instead. For the 1-D Fourier transform, a shift by $a$ is: transform, multiply by the kernel evaluated at $a$, transform back [[1]](#ref-1) (Eq. 69). The **generalized shift** does the same with the DHT kernel [[1]](#ref-1) (Eq. 70):

$$f^{\text{shift}}_{k, k_0} = \sum_{p=1}^{N-1} Y^{nN}_{kp}\, Y^{nN}_{pk_0}\, F_p, \qquad F_p = \sum_{k=1}^{N-1} Y^{nN}_{pk}\, f_k.$$

In words: multiply the transform $F$ by the $k_0$-th column of the kernel, then transform back. The rules below are stated for this dimensionless transform, $F = Y^{nN} f$, without the physical scale factors of `hankel_transform`.

For order 0 there is a picture behind this. In the continuous setting, the generalized shift of a circularly symmetric function by a distance $r_0$ is the average, over every direction, of the function moved by $r_0$ in that direction. A narrow bump at the center therefore comes back, up to a constant factor, as a ring at radius $r_0$. The discrete shift by $k_0$ does the same, with the ring at the sample radius $r_{0k_0}$.

In [ ]:
def generalized_shift(values, k0, Y):
    """Shift a vector by k0 through its own transform: Y (Y[:, k0] * (Y values))."""
    return Y @ (Y[:, k0] * (Y @ values))


# An order-0 kernel with 64 samples, and a narrow bump at the center
n, R, size = 0, 8.0, 64
Y, _ = dht_kernel(order=n, size=size)
r, _ = pypft.sample_points(n=n, size=size, R=R)
bump = np.exp(-((r / 0.3) ** 2))

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 3.8))

# Left: the unshifted bump
axes[0].plot(r, bump, marker="o", markersize=3)
axes[0].set_xlabel(xlabel="$r$")
axes[0].set_title(label="bump at the center")

# Right: its generalized shifts, each landing on the radius r_(k0)
for k0 in (10, 25, 40):
    shifted = generalized_shift(values=bump, k0=k0, Y=Y)
    line = axes[1].plot(r, shifted, label=f"$k_0 = {k0}$")[0]
    axes[1].axvline(x=r[k0], color=line.get_color(), linestyle=":")
axes[1].set_xlabel(xlabel="$r$")
axes[1].set_title(label="generalized shifts (dotted: $r_{0k_0}$)")
axes[1].legend()
fig.tight_layout()
plt.show()

## Rules built on the generalized shift

With the generalized shift in place, the DHT has the same family of rules as the DFT [[1]](#ref-1) (Eqs. 77, 82, 86, 91). Writing $f \Leftrightarrow F$ for a transform pair:

- **Shift-modulation.** A generalized shift in space is a modulation of the transform by a kernel column:

  $$f^{\text{shift}}_{k, k_0} \;\Longleftrightarrow\; Y^{nN}_{mk_0}\, F_m.$$
- **Modulation-shift.** Modulating by a kernel column in space is a generalized shift of the transform:

  $$Y^{nN}_{kk_0}\, g_k \;\Longleftrightarrow\; G^{\text{shift}}_{m, k_0}.$$
- **Convolution.** The convolution built on the generalized shift transforms into a product:

  $$(g * h)_k = \sum_{k_0=1}^{N-1} g_{k_0}\, h^{\text{shift}}_{k, k_0} \;\Longleftrightarrow\; G_m H_m.$$
- **Multiplication.** A product in space transforms into a convolution of the transforms:

  $$g_k h_k \;\Longleftrightarrow\; (G * H)_m = \sum_{q=1}^{N-1} G_q\, H^{\text{shift}}_{m, q}.$$

The cell checks all four at order 2 with random vectors, each rule's two sides computed independently:

In [ ]:
# An order-2 kernel with 32 samples, two random vectors, and their transforms
n, size = 2, 32
Y, _ = dht_kernel(order=n, size=size)
g = rng.standard_normal(size=size)
h = rng.standard_normal(size=size)
G, H = Y @ g, Y @ h
k0 = 7

# Shift-modulation: the transform of a shifted vector
shift_modulation = Y @ generalized_shift(values=g, k0=k0, Y=Y) - Y[:, k0] * G

# Modulation-shift: the transform of a modulated vector
modulation_shift = Y @ (Y[:, k0] * g) - generalized_shift(values=G, k0=k0, Y=Y)

# Convolution: sum of g_(k0) times h shifted by k0, over every k0
h_shifts = np.column_stack(
    tup=[generalized_shift(values=h, k0=j, Y=Y) for j in range(size)]
)
convolution = Y @ (h_shifts @ g) - G * H

# Multiplication: sum of G_q times H shifted by q, over every q
H_shifts = np.column_stack(
    tup=[generalized_shift(values=H, k0=q, Y=Y) for q in range(size)]
)
multiplication = Y @ (g * h) - H_shifts @ G

for name, residual in (
    ("shift-modulation", shift_modulation),
    ("modulation-shift", modulation_shift),
    ("convolution", convolution),
    ("multiplication", multiplication),
):
    print(f"{name:>16}: max residual {np.abs(residual).max():.1e}")

## Where to go next

[04_domains](04_domains.ipynb) introduces the four domains a signal passes through on its way through the polar Fourier transform, in which the DHT is the step from the spatial to the frequency side.

## References

1. <a id="ref-1"></a> N. Baddour, "The Discrete Hankel Transform," in *Fourier Transforms - Century of Digitalization and Increasing Expectations*, IntechOpen, 2019. [doi:10.5772/intechopen.84399](https://doi.org/10.5772/intechopen.84399)